In [2]:
import tensorflow as tf
from tensorflow import keras

In [3]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.metrics

# discontinued import tensorflow_addons as tfa
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.models import load_model

In [ ]:
data_dir = '../input/labeled-optical-coherence-tomography-oct/Dataset - train+val+test'
data = tf.keras.preprocessing.image_dataset_from_directory(data_dir)

In [ ]:
base_dir = os.path.join("../input/labeled-optical-coherence-tomography-oct/Dataset - train+val+test/")
print('Base directory --> ', os.listdir(base_dir))

In [ ]:
# NOT EXCHANGED

train_dir = os.path.join(base_dir + "train/")
print("Train Directory --> ", os.listdir(train_dir))

validation_dir = os.path.join(base_dir + "val/")
print("Validation Directory --> ", os.listdir(validation_dir))

test_dir = os.path.join(base_dir + "test/")
print("Test Directory --> ", os.listdir(test_dir))

In [ ]:
from tensorflow.keras.applications import InceptionV3
inc = InceptionV3(input_shape=(224,224,3),weights='imagenet',include_top=False)
for i in inc.layers:
    i.trainable = False
print(inc.summary())

In [ ]:
model = tf.keras.models.Sequential([
    inc,
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(4, activation = 'softmax')
])

In [ ]:
metrics = ['accuracy',
                tf.keras.metrics.AUC(),
               # tfa.metrics.CohenKappa(num_classes = 4),
               # tfa.metrics.F1Score(num_classes = 4),
                tf.keras.metrics.Precision(), 
                tf.keras.metrics.Recall()]
model.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = metrics)
print(model.summary())

In [10]:
train_datagen = ImageDataGenerator(rescale = 1./255)
train_generator = train_datagen.flow_from_directory(
    train_dir, target_size = (224,224), class_mode = 'categorical', 
    batch_size = 500)

Found 76515 images belonging to 4 classes.


In [11]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir, target_size = (224,224), class_mode = 'categorical', shuffle=False, 
    batch_size = 50)

Found 10933 images belonging to 4 classes.


In [12]:
validation_datagen = ImageDataGenerator(rescale = 1./255)
validation_generator = validation_datagen.flow_from_directory(
    validation_dir, target_size = (224,224), class_mode = 'categorical', 
    batch_size = 16)

Found 21861 images belonging to 4 classes.


In [13]:
# history = model.fit(
#     train_generator,
#     steps_per_epoch = (76515//500),
#     epochs = 30,
#     validation_data = validation_generator,
#     validation_steps = (10933//16),
#     #max_queue_size=100,
#     #workers = 4 ,
#     #verbose = 1
#     )

In [14]:
model.save('/kaggle/working/my_model.h5')

In [15]:
model = load_model('/kaggle/working/my_model.h5')